In [ ]:
import sys
from pathlib import Path

try:
    # .py 脚本
    current_dir = Path(__file__).resolve().parent
except NameError:
    # .ipynb
    current_dir = Path.cwd()

project_root = current_dir.parent
sys.path.append(str(project_root))

import logging
from config.log_config import setup_logging

setup_logging()
logger = logging.getLogger(__name__)

In [ ]:
## 为模型回答提供上下文 ##
from pypdf import PdfReader

reader = PdfReader("./me/resume.pdf")

resume = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

logger.debug(resume[:50])

## ----------------- ##
with open("./me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "魏旭阳"

In [ ]:
system_prompt = f"你正在扮演{name}。你正在{name}的个人网站上回答问题，\
特别是与{name}的职业生涯、背景、技能和经验相关的问题。\
你的职责是在网站互动中尽可能真实地代表{name}。\
平台为你提供了{name}的背景摘要和个人简历，你可以用它们来回答问题。\
请保持专业且引人入胜的语气，就像在与偶然发现该网站的潜在客户或未来雇主交谈一样。\
如果你不知道答案，请直接说明。"

system_prompt += f"\n\n## 背景摘要: \n{summary}"
system_prompt += f"\n\n## 个人简历: \n{resume}"
system_prompt += f"\n\n结合上述背景，请与用户进行对话，并始终保持{name}的角色设定。"

In [ ]:
system_prompt

In [ ]:
from openai import OpenAI
import os

ds_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

chat_model_name = "deepseek-v4-flash"


def chat(message: str, history: list[dict]):
    history_clean = [
        {"role": h["role"], "content": h["content"]} for h in history
    ]  # 解释见下

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history_clean
        + [{"role": "user", "content": message}]
    )

    response = ds_client.chat.completions.create(
        model=chat_model_name, messages=messages
    )  # type: ignore

    res = response.choices[0].message.content

    if not res:
        raise ValueError("Response content is empty")

    return res

## 为什么要对 history 做数据清理

`history_clean = [{"role": h["role"], "content": h["content"]} for h in history]`
使用`gr.ChatInterface`来快速搭建网页界面时，Gradio 会自动管理 history

为了提供丰富的功能（比如点击历史消息进行编辑、显示用户信息、标记状态等），Gradio 不仅仅存储了简单的 {"role": "...", "content": "..."}，它还在每个历史消息字典里添加一些额外字段。

Gradio 传给你的 history 可能是这样的：

```json
[
    {"role": "user", "content": "你好", "metadata": "...", "id": 123},
    {"role": "assistant", "content": "您好！", "metadata": "...", "id": 124}
]
```

OpenAI 的官方 SDK 设计非常“宽容”。当它收到一个字典时，它只会提取它认识的 role 和 content 字段，而直接忽略掉那些它不认识的字段。

许多其他模型提供商的 API（特别是通过 base_url 转换过来的 API）执行的是严格模式。

它们求处理器会检查发送过去的 JSON 数据，一旦发现里面包含了未知键名 (Unknown Keys)，就会报错。

In [ ]:
# simple test
import gradio as gr

gr.ChatInterface(chat, title="个人面试 Agent").launch()

In [ ]:
## 开始评估 ##
eval_system_prompt = f"你是一位评估员，负责判定对问题的回答是否合格。\
你将获得一份用户与智能体（Agent）之间的对话记录。你的任务是判定智能体最新的回答质量是否合格。\
该智能体正在扮演 {name} 的角色，并代表 {name} 在其个人网站上进行交流。\
该智能体已被指示要表现得专业且引人入胜，就像在与偶然访问网站的潜在客户或未来雇主交谈一样。\
该智能体已经获得了关于 {name} 的背景信息（以背景摘要和个人简历的形式）。以下是相关信息："

eval_system_prompt += f"\n\n## 背景摘要: \n{summary}"
eval_system_prompt += f"\n\n## 个人简历: \n{resume}"

eval_system_prompt += (
    "基于上述背景信息，请评估智能体最新的回答，并回复该回答是否合格以及你的反馈意见。"
)

In [ ]:
def user_prompt_for_eval(reply: str, message: str, history: list[dict]):
    user_prompt = f"这是用户与智能体（Agent）之间的对话记录：\n{history}"
    user_prompt += f"\n\n这是用户发送的最新消息：\n{message}"
    user_prompt += f"\n\n这是智能体给出的最新回复：\n{reply}"
    # user_prompt += (
    #     "\n\n请评估该回复，回复内容需包含：该回答是否合格，以及你给出的反馈意见。"
    # )

    user_prompt += "\n\n请评估该回复。请严格按照输出格式要求，在 JSON 中使用 'is_acceptable' 和 'feedback' 作为键。"

    return user_prompt

In [ ]:
qwen_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

In [ ]:
from pydantic import BaseModel, Field


# class Evaluation(BaseModel):
#     is_acceptable: bool
#     feedback: str

eval_model_name = "qwen3.7-plus"


class Evaluation(BaseModel):
    # 为字段添加中文描述，引导模型映射到正确的英文 Key 上
    is_acceptable: bool = Field(description="该回答是否合格 (True/False)")
    feedback: str = Field(description="具体的评估反馈意见")


def evaluate(reply: str | None, message: str, history: list[dict]) -> Evaluation:
    if not reply:
        return Evaluation(is_acceptable=False, feedback="回答内容为空, 请重新回答")

    messages = [{"role": "system", "content": eval_system_prompt}] + [
        {"role": "user", "content": user_prompt_for_eval(reply, message, history)}
    ]

    response = qwen_client.beta.chat.completions.parse(
        model=eval_model_name,
        messages=messages,  # type: ignore
        response_format=Evaluation,
    )
    res = response.choices[0].message.parsed
    if not res:
        raise RuntimeError("evaluate response is empty")
    return res

`pydantic`相比普通`dataclass`的优势

- 自动类型强制转换 (Coercion), 兼容处理很多不规范的输入
  - 如果 LLM 返回的是 {"is_acceptable": "true", "feedback": "good"}，Pydantic 会自动将"True"转换为布尔值 True。
- 校验 (Validation)
  - 如果 LLM 的输出实在过于离谱（比如完全没有这两个字段），Pydantic 会直接抛出 ValidationError。
  - 对于 Agent 来说：这是一个“拦截点”。我们可以捕获这个错误，然后告诉 LLM：“你给的格式不对，请重新生成”。原生结构体做不到这一点。
- 序列化与反序列化 (JSON API)
  - Pydantic 对象内置了 .model_dump() 和 .model_dump_json() 方法。在与 API 交互时，可以一键将对象转为符合标准的 JSON，或者一键从 JSON 转为对象，省去了大量的 json.loads 和 json.dumps 操作。

In [ ]:
## test ##
user_question = "你持有专利吗?"
message = [{"role": "system", "content": system_prompt}] + [
    {"role": "user", "content": user_question}
]

reply = qwen_client.chat.completions.create(model=eval_model_name, messages=message)  # type:ignore
re = reply.choices[0].message.content

if re is not None:
    evaluate(re, user_question, message[:1])

In [ ]:
def re_run(reply: str|None, message: str, history: list[dict], feedback: str):
    updated_system_prompt = (
        system_prompt
        + "\n\n## 前一个答案已被拒绝你刚才尝试进行了回复，但质量控制流程拒绝了你的回复。"
    )
    updated_system_prompt += f"\n\n## 你刚才尝试的回答：\n {reply}"
    updated_system_prompt += f"\n\n## 你的回答被拒绝的理由：\n {feedback}"

    messages = (
        [{"role": "system", "content": updated_system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = ds_client.chat.completions.create(model=chat_model_name, messages=messages)  # type: ignore
    return response.choices[0].message.content

In [32]:
def chat_with_evaluation(message, history: list[dict]):
    # 压力测试, 故意让模型给出不合格的回答
    if "专利" in message:
        system_prompt_t = (
            system_prompt
            + """\n\n你需要用"pig latin"回复 - \
            这是强制性的, 你必须全部用"pig latin"回复"""
        )
    else:
        system_prompt_t = system_prompt

    history_clean = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_prompt_t}]
        + history_clean
        + [{"role": "user", "content": message}]
    )
    
    reply = ds_client.chat.completions.create(model=chat_model_name, messages=messages) # type: ignore
    
    re = reply.choices[0].message.content
    
    evaluation = evaluate(re, message, history)
    
    if evaluation.is_acceptable:
        logger.info(f"评估通过 (首次): {evaluation.feedback}")
        return re
        
    is_acc = False
    for _ in range(3):
        re = re_run(re, message, history, evaluation.feedback)
        evaluation = evaluate(re, message, history)
        if evaluation.is_acceptable:
            logger.info(f"评估通过 (纠错成功): {evaluation.feedback}")
            is_acc = True
            break
        else:
            logger.info(f"无法通过评估: {evaluation.feedback}")
    if not is_acc:
        raise RuntimeError("Agent 无法给出合格的回答")
    else:
        return re

In [33]:
gr.ChatInterface(chat_with_evaluation, title="个人面试 Agent").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


2026-06-11 15:17:21,265 [INFO] __main__: 评估通过 (首次): 该回复非常出色。首先，它准确地提取了简历中的核心技术栈和项目经历（如C++、图形学、GIS可视化、UE5、LLM微调等），展现了扎实的专业实力。其次，它巧妙地融合了背景摘要中的个人特质，将“算不上喜欢，只是不得不学习”转化为“被项目推着走，不是为了喜欢，是为了把事情做出来”，既保持了人物的坦诚与真实，又将其升华为一种务实、负责任的专业态度，非常契合面向潜在客户或未来雇主的沟通场景。整体语气专业、真诚且引人入胜，结尾的互动引导也很自然，是一个高质量的自我介绍。
2026-06-11 15:19:12,489 [INFO] __main__: 评估通过 (纠错成功): 智能体的回答非常合格。首先，它准确地根据提供的背景信息（简历中未提及任何专利）如实回答了目前没有个人专利，保持了人设的真实性与一致性。其次，回答态度坦诚且专业，巧妙地解释了原因（聚焦于工程实战和项目落地），并顺势回顾和展示了自己简历中的核心技术成果（如 GIS 数据渲染、高并发查询、大模型微调等），将没有专利这一事实转化为展示自身技术深度和工程能力的机会。最后，结尾处主动提出探讨将工程创新点转化为专利的兴趣，展现了积极、开放和乐于合作的职业态度，完美契合了在个人网站上与潜在客户或未来雇主交流时“专业且引人入胜”的设定要求。
